[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week13.ipynb)

# Week 13: Integration & Transfer Learning
**Introduction to Deep Learning (HIT)** &middot; Part IV: Integration

Transfer learning and fine-tuning; model inference; the end-to-end workflow into the advanced courses.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Fine-tune a pretrained model end-to-end.
- Run inference and assemble a full workflow.
- Reason about when transfer learning helps.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. A pretrained model with a new head
Load ResNet-18 trained on ImageNet, replace the final layer for 5 classes (downloads ~45 MB).

In [ ]:
from torchvision import models
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print("original final layer:", net.fc)
net.fc = nn.Linear(net.fc.in_features, 5)        # new 5-class head
print("new final layer:", net.fc)
print("total params:", sum(p.numel() for p in net.parameters()))

## 2. The preprocessing must match the pretrained model
ResNet expects 3x224x224, normalized with ImageNet statistics.

In [ ]:
from torchvision import transforms
prep = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
])
print("preprocessing pipeline:\n", prep)

## 3. Three regimes, by trainable-parameter count
From-scratch trains everything; frozen trains only the head; fine-tune trains all from pretrained.

In [ ]:
def trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

frozen = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in frozen.parameters(): p.requires_grad = False
frozen.fc = nn.Linear(frozen.fc.in_features, 5)
print("frozen-features trainable:", f"{trainable(frozen):,}")

ft = models.resnet18(weights=models.ResNet18_Weights.DEFAULT); ft.fc = nn.Linear(ft.fc.in_features, 5)
print("fine-tune trainable:     ", f"{trainable(ft):,}")

scratch = models.resnet18(weights=None); scratch.fc = nn.Linear(scratch.fc.in_features, 5)
print("from-scratch trainable:  ", f"{trainable(scratch):,}")

## 4. Feature extraction: the backbone as an encoder
Run images through the frozen backbone to get embeddings, then classify them.

In [ ]:
backbone = nn.Sequential(*list(frozen.children())[:-1])   # everything except the final fc
backbone.eval()
batch = torch.randn(4, 3, 224, 224)                       # 4 preprocessed images
with torch.no_grad():
    emb = backbone(batch).flatten(1)
print("embeddings:", tuple(emb.shape), "(512-d per image) -> feed a linear classifier")

## 5. A linear probe on the embeddings
Train only a small head on the frozen 512-d features.

In [ ]:
torch.manual_seed(0)
feat = torch.randn(60, 512); lab = torch.randint(0, 5, (60,))   # stand-in for real embeddings + labels
head = nn.Linear(512, 5); o = torch.optim.Adam(head.parameters(), 0.01); f = nn.CrossEntropyLoss()
for _ in range(200):
    o.zero_grad(); f(head(feat), lab).backward(); o.step()
print("linear-probe train accuracy:", round((head(feat).argmax(1) == lab).float().mean().item(), 3))

## 6. Layer-wise learning rates
Fine-tuning usually uses a smaller rate on the pretrained backbone than on the new head.

In [ ]:
opt = torch.optim.Adam([
    {"params": ft.fc.parameters(), "lr": 1e-3},                 # new head: faster
    {"params": [p for n, p in ft.named_parameters() if not n.startswith("fc")], "lr": 1e-5},  # backbone: slower
])
print("param groups and their learning rates:", [g["lr"] for g in opt.param_groups])

## 7. Inference end to end
Eval mode, no gradients, softmax to probabilities.

In [ ]:
net.eval()
with torch.no_grad():
    logits = net(torch.randn(1, 3, 224, 224))         # one preprocessed image
    probs = logits.softmax(1)
print("predicted class:", probs.argmax(1).item(), "| confidence:", round(probs.max().item(), 3))

## 8. Save and load a checkpoint
Persist the weights, then restore them, the end of the workflow.

In [ ]:
torch.save(net.state_dict(), "model.pt")
fresh = models.resnet18(weights=None); fresh.fc = nn.Linear(fresh.fc.in_features, 5)
fresh.load_state_dict(torch.load("model.pt"))
print("checkpoint saved and reloaded; ready for inference or further fine-tuning.")

**Mini-exercise:** time a forward pass through the full network vs the frozen backbone only. Feature extraction is cheaper because no gradients flow through the backbone. **Wrap-up:** this end-to-end workflow, data, model, train, evaluate, infer, is exactly what the advanced vision and language courses build on.

---
Student materials for this week: the lab handout (`labs/week13.html`) and the curated references (`references/week13.html`) in the course site.